# break detector 0.870, inference

Loads the saved LightGBM (5 fold models, 0.870 out-of-fold AUC) and scores new series. Nothing is trained here.

Output: `id,break_probability`

In [ ]:
#settings, change these paths
SERIES = 'data/test_series.parquet'          #new series, same format as the competition file
MODEL = 'try2/model'                 #fold0..4.txt + columns.json
REF = 'B:/AI_cpu_087910_20261108/work'       #training folder, has the feature code + train-side reference tables
OUT = 'predict_0870'                         #scratch folder, must be empty
CSV = 'submission.csv'
WORKERS = 2

In [ ]:
#imports
import os, sys, json, subprocess
import pandas as pd
import matplotlib.pyplot as plt
os.environ['CUDA_VISIBLE_DEVICES'] = ''   #cpu only

In [ ]:
#look at the new data
raw = pd.read_parquet(SERIES)
print(raw.shape, raw['id'].nunique(), 'series')
raw.head()

In [ ]:
#plot a few, red line is the boundary we score
ids = raw['id'].unique()[:3]
fig, ax = plt.subplots(len(ids), 1, figsize=(10, 6))
for a, i in zip(ax, ids):
    s = raw[raw['id'] == i].sort_values('time')
    a.plot(s['value'].values, lw=.7)
    a.axvline((s['period'] == 0).sum(), ls='--', c='r')
    a.set_title(str(i))
plt.tight_layout()

In [ ]:
#the saved model
info = json.load(open(f'{MODEL}/model.json'))
print(info['model'], 'oof auc', round(info['oof_auc'], 5))
print(len(json.load(open(f'{MODEL}/columns.json'))), 'features')

In [ ]:
#run it:
#1 raw series -> 977 base features
#2 grid, grid2, diff, likelihood ratio, garch, spectral, harmonic blocks
#3 mechanism tables (periodic, ar, phase, volatility, jump laws, noise tests...)
#4 load fold0..4, average, write id,break_probability
subprocess.run([sys.executable, 'try2/predict_compact.py', '--series', SERIES, '--model', MODEL,
                '--ref', REF, '--out', OUT, '--csv', CSV, '--workers', str(WORKERS)], check=True)

In [ ]:
#check the output
sub = pd.read_csv(CSV)
print(sub.shape)
print(sub.break_probability.describe())
sub.head()